In [1]:
import requests
import json
import uuid
from IPython.display import display_javascript, display_html, display
import pandas as pd

response = requests.get("http://api.openweathermap.org/data/2.5/forecast?id=5781004&APPID=e0c55e00bf021f6142de334823046e9e&units=imperial")
weather = response.content.decode("utf-8")
weatherDict = json.loads(weather)

In [2]:
print(json.dumps(weatherDict, indent=2))

{
  "message": 0.0104,
  "city": {
    "sys": {
      "population": 0
    },
    "coord": {
      "lon": -111.87632,
      "lat": 40.6255
    },
    "country": "US",
    "id": 5781004,
    "population": 0,
    "name": "Salt Lake County"
  },
  "cod": "200",
  "cnt": 39,
  "list": [
    {
      "weather": [
        {
          "description": "scattered clouds",
          "icon": "03n",
          "id": 802,
          "main": "Clouds"
        }
      ],
      "sys": {
        "pod": "n"
      },
      "wind": {
        "speed": 3.6,
        "deg": 354
      },
      "dt_txt": "2016-11-13 03:00:00",
      "main": {
        "humidity": 100,
        "temp": 46.62,
        "temp_max": 46.62,
        "sea_level": 1036.37,
        "temp_min": 45.79,
        "grnd_level": 872.52,
        "temp_kf": 0.46,
        "pressure": 872.52
      },
      "clouds": {
        "all": 36
      },
      "dt": 1479006000
    },
    {
      "weather": [
        {
          "description": "few clouds",
         

In [3]:
# Method for rendering collapsible JSON from: http://stackoverflow.com/questions/18873066/pretty-json-formatting-in-ipython-notebook

class RenderJSON(object):
    def __init__(self, json_data):
        if isinstance(json_data, dict):
            self.json_str = json.dumps(json_data)
        else:
            self.json_str = json
        self.uuid = str(uuid.uuid4())

    def _ipython_display_(self):
        display_html('<div id="{}" style="height: 600px; width:100%;"></div>'.format(self.uuid),
        raw=True)
        
        display_javascript("""
        require(["https://rawgit.com/caldwell/renderjson/master/renderjson.js"], function() {
        document.getElementById('%s').appendChild(renderjson(%s))
        });
        """ % (self.uuid, self.json_str), raw=True)
        
RenderJSON(weatherDict)

In [10]:
#pdWeather = pd.read_json(weatherDict)
#pdWeather

#pd.DataFrame(weatherDict["data"], columns=[x["label"] for x in weatherDict["fields"]])

timeList = []
maxTempList = []
minTempList = []
pressureList = []
tempList = []
wind_speedList = []
windDegList = []
rainList = []
rainListML = []

for weatherEntry in weatherDict["list"]:
    timeList.append(weatherEntry["dt_txt"])
    mainWeather = weatherEntry["main"]
    maxTempList.append(mainWeather["temp_max"])
    minTempList.append(mainWeather["temp_min"])
    pressureList.append(mainWeather["pressure"])
    tempList.append(mainWeather["temp"])
    windWeather = weatherEntry["wind"]
    wind_speedList.append(windWeather["speed"])
    windDegList.append(windWeather["deg"])
    if ("rain" in weatherEntry):
        rainList.append(1)
        rainListML.append(weatherEntry["rain"]["3h"])
    else:
        rainList.append(0)
        rainListML.append(0)
    
    

In [11]:
data = [('DateTime', timeList),
         ('MaxTemp', maxTempList),
         ('MinTemp', minTempList),
         ('Pressure', pressureList),
         ('Temperature', tempList),
         ('WindSpeed', wind_speedList), 
         ('WindDeg', windDegList),
         ('Rain', rainList),
         ('Rain (ml)', rainListML)
         ]
weatherForecast = pd.DataFrame.from_items(data)
weatherForecast

,DateTime,MaxTemp,MinTemp,Pressure,Temperature,WindSpeed,WindDeg,Rain,Rain (ml)
0,2016-11-13 03:00:00,46.62,45.79,872.52,46.62,3.60,354.0000,0,0.00
1,2016-11-13 06:00:00,43.47,42.92,872.62,43.47,3.60,142.5000,0,0.00
2,2016-11-13 09:00:00,41.65,41.37,872.54,41.65,5.57,153.5010,0,0.00
3,2016-11-13 12:00:00,40.36,40.36,873.85,40.36,2.04,170.0070,0,0.00
4,2016-11-13 15:00:00,41.42,41.42,874.45,41.42,4.16,157.5010,0,0.00
5,2016-11-13 18:00:00,50.89,50.89,875.13,50.89,3.74,169.0000,0,0.00
6,2016-11-13 21:00:00,54.33,54.33,874.58,54.33,3.85,319.0010,0,0.00
7,2016-11-14 00:00:00,50.55,50.55,874.11,50.55,2.17,350.5080,0,0.00
8,2016-11-14 03:00:00,44.06,44.06,874.39,44.06,2.80,100.5010,0,0.00
9,2016-11-14 06:00:00,42.56,42.56,874.61,42.56,7.29,153.5040,0,0.00


In [6]:
weatherForecast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 8 columns):
DateTime       39 non-null object
MaxTemp        39 non-null float64
MinTemp        39 non-null float64
Pressure       39 non-null float64
Temperature    39 non-null float64
WindSpeed      39 non-null float64
WindDeg        39 non-null float64
Rain           39 non-null int64
dtypes: float64(6), int64(1), object(1)
memory usage: 2.5+ KB


In [7]:
weatherForecast.describe()

,MaxTemp,MinTemp,Pressure,Temperature,WindSpeed,WindDeg,Rain
count,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.0
mean,43.807436,43.764872,868.345897,43.807436,5.937179,205.348918,0.0
std,5.822974,5.818249,5.994079,5.822974,3.685992,83.846970,0.0
min,33.520000,33.520000,858.680000,33.520000,1.720000,12.501700,0.0
25%,40.200000,40.200000,862.570000,40.200000,3.040000,153.002500,0.0
50%,43.450000,42.920000,870.770000,43.450000,4.470000,169.000000,0.0
75%,48.390000,47.975000,873.980000,48.390000,8.600000,276.501500,0.0
max,55.380000,55.380000,875.130000,55.380000,14.140000,354.000000,0.0
